# Auto-HKG × Unsloth — Local LLM di T4 Colab

> Jalankan Auto-HKG **tanpa API key** menggunakan model lokal via Unsloth.
> GPU: T4 16GB (Colab free tier)

### Pilihan model (semua 4-bit, aman di T4):
| Alias | Model | VRAM | Kecepatan JSON |
|---|---|---|---|
| `qwen2.5-7b` | Qwen2.5-7B-Instruct | ~5GB | ⭐⭐⭐⭐ |
| `qwen2.5-14b` | Qwen2.5-14B-Instruct | ~9GB | ⭐⭐⭐⭐⭐ |
| `qwen3-8b` | Qwen3-8B | ~6GB | ⭐⭐⭐⭐ |
| `llama3.1-8b` | Llama-3.1-8B-Instruct | ~6GB | ⭐⭐⭐ |
| `gemma3-12b` | Gemma-3-12B | ~8GB | ⭐⭐⭐ |
| `deepseek-r1-7b` | DeepSeek-R1-Distill-7B | ~5GB | ⭐⭐⭐⭐ |

In [ ]:
# ── 0. Pastikan GPU aktif ──────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        '❌ GPU tidak terdeteksi!\n'
        'Aktifkan dulu: Runtime → Change runtime type → T4 GPU'
    )
print(f'✅ GPU aktif: {result.stdout.strip()}')

In [ ]:
# ── 1. Install Unsloth ─────────────────────────────────────────────
# Unsloth punya installer khusus — jangan pakai pip install unsloth biasa
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers trl peft accelerate bitsandbytes

In [ ]:
# ── 2. Install dependencies proyek ────────────────────────────────
!pip install -q pandas networkx tqdm matplotlib pyvis

In [ ]:
# ── 3. Clone repo ──────────────────────────────────────────────────
!git clone https://github.com/USERNAME/auto-hkg.git   # ganti USERNAME
%cd auto-hkg
import sys
sys.path.insert(0, 'src')

In [ ]:
# ── 4. Upload dataset ──────────────────────────────────────────────
from google.colab import files
import shutil, os

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
shutil.move(list(uploaded.keys())[0], 'data/Knowledge_Base_Update.csv')
print('✅ Dataset siap')

In [ ]:
# ── 5. Cek VRAM sebelum load model ────────────────────────────────
import torch

total  = torch.cuda.get_device_properties(0).total_memory / 1e9
free   = torch.cuda.mem_get_info()[0] / 1e9
print(f'VRAM total : {total:.1f} GB')
print(f'VRAM bebas : {free:.1f} GB')
print()

# Rekomendasi otomatis berdasarkan VRAM
if free >= 12:
    print('✅ Rekomendasi: qwen2.5-14b (14B parameter, terbaik)')
elif free >= 8:
    print('✅ Rekomendasi: qwen3-8b atau gemma3-12b')
elif free >= 5:
    print('✅ Rekomendasi: qwen2.5-7b atau llama3.1-8b')
else:
    print('⚠️  VRAM terbatas — pakai llama3.2-3b atau gemma3-4b')

In [ ]:
# ── 6. Pilih model & jalankan pipeline ────────────────────────────
#
# Ganti MODEL_ALIAS dengan salah satu:
#   'qwen2.5-7b'    → Qwen2.5-7B-Instruct  (~5GB, recommended)
#   'qwen2.5-14b'   → Qwen2.5-14B-Instruct (~9GB, terbaik)
#   'qwen3-8b'      → Qwen3-8B             (~6GB)
#   'llama3.1-8b'   → Llama-3.1-8B         (~6GB)
#   'gemma3-4b'     → Gemma-3-4B-IT        (~4GB, paling ringan)
#   'gemma3-12b'    → Gemma-3-12B-IT       (~8GB)
#   'deepseek-r1-7b'→ DeepSeek-R1-7B       (~5GB)
#
# Atau pakai HF repo ID langsung:
#   model = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'

MODEL_ALIAS = 'qwen2.5-7b'   # ← GANTI DI SINI

from auto_hkg import AutoHKG, HKGConfig

cfg = HKGConfig(
    provider='unsloth',
    model=MODEL_ALIAS,
    data_path='data/Knowledge_Base_Update.csv',
    batch_size=5,              # lebih kecil karena inferensi lokal lebih lambat
    sleep_between_batches=0.0, # tidak perlu delay — tidak ada rate limit
)

pipeline = AutoHKG(cfg)
pipeline.run()

In [ ]:
# ── 7. Monitor VRAM selama pipeline ───────────────────────────────
# Jalankan cell ini di tab terpisah (Ctrl+M B) sambil pipeline jalan
import torch, time

for _ in range(20):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f'VRAM used: {used:.2f}GB | reserved: {reserved:.2f}GB')
    time.sleep(10)

In [ ]:
# ── 8. Benchmark: bandingkan 2 model (opsional) ───────────────────
# Test cepat output JSON dari beberapa model sebelum full run

import sys, json, time
sys.path.insert(0, 'src')
from llm_client import LLMClient

TEST_PROMPT = '''Kamu adalah sistem ekstraksi knowledge graph pendidikan.
Konteks : Lokasi absolut adalah letak tetap berdasarkan garis lintang dan bujur.
Pertanyaan: Apa yang dimaksud lokasi absolut?
Level Kognitif: C1-Mengingat

Kembalikan HANYA JSON:
{"topic_coarse": "...", "topic_fine": "...", "concepts": [...],
 "methods": [...], "bloom_level": "C1-Mengingat",
 "prerequisites": [...], "successors": [...], "difficulty": 1}'''

for alias in ['qwen2.5-7b', 'llama3.1-8b']:   # tambah alias lain jika mau
    print(f'\n── Testing {alias} ──')
    try:
        client = LLMClient(provider='unsloth', model=alias)
        t0 = time.time()
        raw = client.complete(TEST_PROMPT)
        elapsed = time.time() - t0
        parsed = json.loads(raw)
        print(f'  ✅ JSON valid | {elapsed:.1f}s | concepts: {parsed["concepts"]}')
    except json.JSONDecodeError:
        print(f'  ⚠️  Output bukan JSON valid:\n  {raw[:200]}')
    except Exception as e:
        print(f'  ❌ Error: {e}')

    # Bersihkan VRAM sebelum load model berikutnya
    import torch, gc
    del client
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ── 9. Visualisasi & download ──────────────────────────────────────
from visualize_graph import load_graph, print_stats, plot_static, plot_interactive
from IPython.display import Image

G = load_graph()
print_stats(G)
plot_static(G)
Image('output/graph/graph_static.png')

In [ ]:
# ── 10. Download output ────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('auto_hkg_output', 'zip', 'output')
files.download('auto_hkg_output.zip')